In [0]:
%sql
CREATE CATALOG IF NOT EXISTS bootcamp_oscar;

SHOW CATALOGS;

## Paso 2: Crear los esquemas Landing y Bronze

- El esquema `landing` almacenará los archivos crudos (CSV) — simula el data lake
- El esquema `bronze` contendrá las tablas crudas sin procesar


In [0]:
%sql
USE CATALOG bootcamp_oscar;
CREATE SCHEMA IF NOT EXISTS bootcamp_oscar.landing
COMMENT 'Esquema para archivos Crudos (simula data lake)';

CREATE SCHEMA IF NOT EXISTS bootcamp_oscar.bronze
COMMENT ' Esquema para datos crudos sin procesar (Bronxe layer) ';

-- Verificamos
SHOW SCHEMAS;
 


## Paso 3: Crear el Volume para archivos

Un **Volume** es donde subiremos nuestro archivo CSV.

In [0]:
%sql
-- Crear Volume para archivos
-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp_oscar.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';

-- Verificar que se creo correctamente 
SHOW VOLUMES in bootcamp_oscar.landing;

In [0]:
%sql
--Verificamos que archivos esten subidos 
LIST '/Volumes/bootcamp_oscar/landing/archivos';

## Paso 5: Crear la tabla properties_bronze

Ahora vamos a crear una tabla que lea los datos del CSV.

**Tipo de tabla:** EXTERNAL (los datos quedan en el Volume, si borramos la tabla el CSV sigue existiendo)


In [0]:
%sql
-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE  bootcamp_oscar.bronze.properties_bronze
  SELECT * FROM read_files(
    '/Volumes/bootcamp_oscar/landing/archivos/properties_raw.csv',
  format => 'csv',
  header => true
  )
  WHERE url like 'https%';

In [0]:
%sql
--verificamos que exista la tabla
Show TABLES IN bootcamp_oscar.bronze;

In [0]:
%sql
DESC bootcamp_oscar.bronze.properties_bronze;

In [0]:
%sql
SELECT COUNT(*) FROM bootcamp_oscar.bronze.properties_bronze;

In [0]:
%sql
-- SELECT CON Columnas importantes
SELECT id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes,
metros_cuadrados_totales, antiguedad, estado, zona
FROM bootcamp_oscar.bronze.properties_bronze
LIMIT 10;

Contar nulos por columna - ¿Cuántos valores nulos hay en cada columna importante?
Calcula para: precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales,
metros_cuadrados_cubiertos, orientacion_cardinal, piso, cochera, antiguedad, estado, zona.
Pista:
Usa COUNT(*) - COUNT(columna) para cada columna. Puedes usar CTEs para organizar.

In [0]:
%sql
-- Nulos para las columnas con CTEs

WITH total AS(
  SELECT count(*) AS n
  FROM bootcamp_oscar.bronze.properties_bronze
),
nulos AS(
  SELECT COUNT(*) - COUNT(precio) AS Nulos_precio,
  COUNT(*) -count(expensas) AS Nulos_expensas,
  COUNT(*) - count(tipo_de_operacion) AS Nulos_tpo_operacion,
  COUNT(*) - count(moneda) AS Nulos_moneda,
  COUNT(*) - count(ambientes) AS Nulos_ambientes,
  COUNT(*) - count(metros_cuadrados_totales) AS Nulos_metros_totales,
  COUNT(*) - count(metros_cuadrados_cubiertos) AS Nulos_metros_cubiertos,
  COUNT(*) - count(orientacion_cardinal) AS Nulos_orientacion,
  COUNT(*) - count(piso) AS Nulos_piso,
  COUNT(*) - count(cochera) AS Nulos_cochera,
  COUNT(*) - count(antiguedad) AS Nulos_antiguedad,
  COUNT(*) - count(estado) AS Nulos_estado,
  COUNT(*) - count(zona) AS Nulos_zona
  FROM bootcamp_oscar.bronze.properties_bronze
)
SELECT t.n,
n.*
FROM total t, nulos n;


In [0]:
%sql
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================
-- Usamos CTE para calcular el total y los nulos por columna

WITH total AS (
    SELECT COUNT(*) as n 
    FROM bootcamp_oscar.bronze.properties_bronze
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(ubicacion) as nulos_ubicacion,
        COUNT(*) - COUNT(precio) as nulos_precio,
        COUNT(*) - COUNT(expensas) as nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) as nulos_tipo_operacion,
        COUNT(*) - COUNT(moneda) as nulos_moneda,
        COUNT(*) - COUNT(ambientes) as nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) as nulos_m2_totales,
        COUNT(*) - COUNT(metros_cuadrados_cubiertos) as nulos_m2_cubiertos,
        COUNT(*) - COUNT(orientacion_cardinal) as nulos_orientacion_cardinal,
        COUNT(*) - COUNT(orientacion_inmueble) as nulos_orientacion_inmueble,
        COUNT(*) - COUNT(piso) as nulos_piso,
        COUNT(*) - COUNT(cochera) as nulos_cochera,
        COUNT(*) - COUNT(antiguedad) as nulos_antiguedad,
        COUNT(*) - COUNT(estado) as nulos_estado,
        COUNT(*) - COUNT(zona) as nulos_zona
    FROM bootcamp_oscar.bronze.properties_bronze
)
SELECT 
    t.n as total_registros,
    n.*
FROM total t, nulos n;


In [0]:
%sql
-- ============================================================
-- EDA 2.2: Porcentaje de nulos en columnas clave
-- ============================================================

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp_oscar.bronze.properties_bronze
)
SELECT 
    ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as pct_nulos_precio,
    ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as pct_nulos_expensas,
    ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as pct_nulos_ambientes,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
FROM bootcamp_oscar.bronze.properties_bronze
CROSS JOIN totales t
GROUP BY t.total;